# Topological Approach for Data Assimilation (TADA) Mini Tutorial

## TDA Seminar Michigan State University 
## 4/9/25
### By: Max Chumley
<div style="text-align: center;">
    <img src="https://www.maxchumley.com/assets/images/TADA_anim.gif" alt="TADA Workflow" style="width:50%; height:auto;">
</div>

Welcome to this tutorial on the Topological Approach for Data Assimilation (TADA) algorithm. If at any point you get stuck, the answer key has been provided in markdown cells throughout this notebook. If you double click those cells you will see the commented out answers. 

## Part I: Persistence Optimization (15 minutes)

For the first tutorial, we will study persistence optimization by optimizing the positions of points in a point cloud to minimize some persistence based loss function. Specifically, we will aim to minimize the Wasserstein distance between "model" and "measurement" persistence diagrams to minimize topological differences between the two point clouds to visualize the optimization process. Run the cell below to import the necessary libraries below and if any fail to import make sure you install them. 

In [ ]:
# Import necessary libraries
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import gudhi as gd
from gudhi.tensorflow import RipsLayer
from matplotlib.gridspec     import GridSpec
import os
from gudhi.wasserstein import wasserstein_distance
from teaspoon.MakeData.PointCloud import Annulus 
from IPython.display import clear_output

Start by initializing "model" and "measurement/target" point clouds. I recommend using the ```Annulus``` function from [teaspoon](https://teaspoontda.github.io/teaspoon/modules/MakeData/PointCloud.html) to generate points cloud an annulus with additive noise. Fill in the blanks below for ```X_model``` and ```X_target``` and make two point clouds each with 50 points. 

For ```X_model``` set ```r=0.5``` and ```R=0.51``` and set ```X_target``` to be half of ```X_model``` so it gives a circle with half the radius. 

### Answer Key (double click this cell):

---

<!-- X_model = Annulus(N=50, r=0.5, R=0.51) -->

<!-- X_target = 0.5*X_model -->

---

In [ ]:
# Generate point cloud of random points
np.random.seed(1)

X_model = # YOUR CODE HERE
X_target = # YOUR CODE HERE


######################################################################
######################################################################
######################################################################
# Plot the point clouds
plt.figure()
plt.scatter(X_model[:,0], X_model[:,1], label='Initial')
plt.scatter(X_target[:,0], X_target[:,1], label='Target', color='red')
plt.title('Point cloud at epoch 0')
plt.legend()
plt.show()

Next, we will use the ```gudhi``` python library to plot the persistence diagrams for the model point cloud. Run the cell below to plot the persistence diagrams for ```X_model``` and ```X_target```. What is different about the two diagrams?

In [ ]:
# Plot starting persistence diagram

st = gd.RipsComplex(points=X_model).create_simplex_tree(max_dimension=2)
dgm = st.persistence()
plot = gd.plot_persistence_diagram(dgm)
plot.set_title("Persistence Diagram for X_model")

st2 = gd.RipsComplex(points=X_target).create_simplex_tree(max_dimension=2)
dgm2 = st2.persistence()
plot = gd.plot_persistence_diagram(dgm2)
plot.set_title("Persistence Diagram for X_target")


Notice the topological differences in the persistence diagrams. This difference can be quantified by the Wasserstein distance. Run the cell below to initialize the optimization function, but do not change any of the code for now. This function takes in two point clouds ```X_model``` and ```X_target``` and a predefined loss function in terms of persistence diagrams along with optimization parameters to minimize the loss function. 

---

### DO NOT EDIT

In [ ]:
# This function is used for performing the optimization with gudhi. Feel free to look at what it is doing, but DO NOT EDIT IT until you have completed part 1 and have extra time. 
def optimize_persistence(X, X_target, loss_fn, n_epochs, initial_lr, include_residual_loss=False):
    # Define optimizer and hyperparameters
    lr = tf.keras.optimizers.schedules.InverseTimeDecay(initial_learning_rate=initial_lr, decay_steps=100, decay_rate=.01)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    X = tf.Variable(X, dtype=tf.float64)
    X_target = tf.constant(X_target, dtype=tf.float64)
    # Perform optimization
    losses, dgms = [], []
    layer = RipsLayer(homology_dimensions=[0,1])
    target_dgm = layer.call(tf.cast(X_target,dtype=np.float32))
    target_dgm0 = tf.cast(target_dgm[0][0], dtype=np.float64)
    target_dgm1 = tf.cast(target_dgm[1][0], dtype=np.float64)

    for epoch in range(n_epochs+1):
        with tf.GradientTape() as tape:
            dgm = layer.call(tf.cast(X,dtype=np.float32))
            dgm0 = tf.cast(dgm[0][0], dtype=np.float64)
            dgm1 = tf.cast(dgm[1][0], dtype=np.float64)
            
            persistence_loss =  loss_fn(dgm1, target_dgm1)
            
            if include_residual_loss:
                residuals = X-X_target
                res_dgm = layer.call(tf.cast(residuals,dtype=np.float32))
                res_dgm0 = tf.cast(res_dgm[0][0], dtype=np.float64)
                res_dgm1 = tf.cast(res_dgm[1][0], dtype=np.float64)
                empty_dgm = tf.constant([], dtype=np.float64)
                residual_loss = loss_fn(res_dgm0, empty_dgm) + loss_fn(res_dgm1, empty_dgm)
            else:
                residual_loss = 0
            # Unit square regularization
            regularization = tf.reduce_sum(tf.maximum(tf.abs(X)-1, 0)) + residual_loss
            loss =  persistence_loss + regularization
        gradients = tape.gradient(loss, [X])
        
        # Apply small random noise to the gradient to ensure convergence
        # np.random.seed(epoch)
        
        optimizer.apply_gradients(zip(gradients, [X]))
        print(loss.numpy())
        losses.append(loss.numpy())
        dgms.append(dgm)
        
        ####################################################################################
        # Plot the current step
        # Create a 1x3 grid
        clear_output(wait=True)
        fig = plt.figure(figsize=(12, 4), dpi=100)
        grid = GridSpec(1, 3, width_ratios=[1, 1, 1])  # 1 row, 3 columns

        # Create subplots in each grid cell
        ax1 = plt.subplot(grid[0, 0])
        ax2 = plt.subplot(grid[0, 1])
        ax3 = plt.subplot(grid[0, 2])

        # Plot point cloud
        ax1.scatter(X.numpy()[:,0], X.numpy()[:,1], label='Model', color='blue')
        ax1.scatter(X_target.numpy()[:,0], X_target.numpy()[:,1], label='Measurement', color='red', marker='x')
        ax1.set_xlim(-1.1, 1.1)
        ax1.set_ylim(-1.1, 1.1)
        ax1.set_title('Point Cloud')
        ax1.legend(loc='upper right')

        # Plot persistence diagram
        birth, death = [], []
        for pair in dgm[1][0].numpy():
            birth.append(pair[0])
            death.append(pair[1])
        ax2.scatter(birth, death, c='blue')
        birth, death = [], []
        for pair in target_dgm[1][0].numpy():
            birth.append(pair[0])
            death.append(pair[1])

        ax2.scatter(birth, death, c='red', marker='x')
        ax2.plot([-0.1, 2], [-0.1, 2], color='black')
        ax2.set_xlabel("Birth")
        ax2.set_ylabel("Death")
        ax2.set_xlim(0, 2)
        ax2.set_ylim(0, 2)

        # Plot loss
        if epoch == 0:
            max_loss = loss.numpy()
        ax3.plot(losses, color='blue', label='Plot 3')
        ax3.set_title('Loss')
        ax3.set_xlim(0, n_epochs)
        ax3.set_ylim(0,max_loss)
        plt.tight_layout()
        plt.show()
        plt.close(fig)
        ####################################################################################

### DO NOT EDIT

---

We need to define the loss function now. I have given you a skeleton for doing this. Use the ```wasserstein_distance``` function from [link](https://gudhi.inria.fr/python/latest/wasserstein_distance_user.html). Make sure you set ```order=1``` and ```internal_p=2``` for this example and set ```enable_autodiff=True``` to allow for computing the gradient with ```tensorflow```. This format is used so we can easily change the loss function if we want and add more objectives.

### Answer Key (double click this cell):

---

<!-- wasserstein_dist = wasserstein_distance(dgm, target_dgm, order=1,internal_p=2,enable_autodiff=True) -->

---

In [ ]:

def loss_fn(dgm, target_dgm):
    # Compute the Wasserstein distance between the current persistence diagram and the target
    wasserstein_dist = # YOUR CODE HERE
    
    return wasserstein_dist


Now run the optimize persistence function to observe how the model point cloud is updated to minimize the Wasserstein distance between the model and target persistence diagrams. Note: we are only minimizing the Wasserstein distance between 1D persistence diagrams here but we will use 0D persistence later.

In [ ]:
optimize_persistence(X_model, X_target, loss_fn, n_epochs=100, initial_lr=0.01, include_residual_loss=False)

Notice that the model point cloud is updated to a topologically equivalent point cloud, but because the target point cloud is inside of the model point cloud only a few points needed to move to minimize the loss function. Next, try the same optimization problem, but change $X_{target}=2X_{model}$.

In [ ]:
optimize_persistence(X_model, 2*X_model, loss_fn, n_epochs=300, initial_lr=0.01, include_residual_loss=False)

We still reach a minimizer in this case and the updated point cloud much more closely matches the measurements in terms of topology, but we have completely lose any temporal data if these point clouds are the state space of some dynamical system. Re-run the previous example now, but in ```optimize_persistence``` set ```include_residual_loss``` to ```True```. The loss function is now minimizing the Wasserstein distance of the residual persistence diagrams in addition to the original Wasserstein distance term. This ensures a proper matching between points and prevents topologically equivalent shifts.

In [ ]:
optimize_persistence(X_model, 2*X_model, loss_fn, n_epochs=300, initial_lr=0.01, include_residual_loss=True)

Notice that each point is attracted to another point in the measurement point cloud and the optimization process gives a much better prediction of the measurements in this case. Finally, run the next cell using the original ```X_target``` point cloud to see that it works much better in reverse now. 

In [ ]:
optimize_persistence(X_model, X_target, loss_fn, n_epochs=150, initial_lr=0.01, include_residual_loss=True)

If you still have time, try to modify the point cloud in some way and try to optimize for a different case.

# STOP HERE 🛑

---

## Part II: Time Series Forecasting (15 minutes)

Now we will use the random feature map forecasting method to generate dynamical system forecasts with teaspoon. 

In [ ]:
# Import necessary libraries 
import numpy as np
from matplotlib import rc
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from teaspoon.MakeData.DynSysLib.autonomous_dissipative_flows import lorenz
from teaspoon.DAF.forecasting import random_feature_map_model
from teaspoon.DAF.forecasting import get_forecast
r_seed = 48824
np.random.seed(r_seed)

First we need to generate time varying data to use for the forecast. For this we will use the [dynamic systems library](https://teaspoontda.github.io/teaspoon/modules/MakeData/DynSysLib/index.html) in ```teaspoon``` to simulate data from the chaotic Lorenz system.

In [ ]:
# Simulate the Lorenz system at a random initial condition
ICs = list(np.random.normal(size=(3,1)).reshape(-1,))
t, ts = lorenz(L=500, fs=50, SampleSize=6001, parameters=[28,10.0,8.0/3.0],InitialConditions=ICs)
ts = np.array(ts)

Because we are dealing with measurement data, we need to artificially inject noise into the signals to contaminate the training data. 

In [ ]:
train_len = 4000
forecast_len = 300

# Add noise to signals
noise = np.random.normal(scale=0.01, size=np.shape(ts[:,0:train_len+forecast_len]))
u_obs = ts[:,0:train_len+forecast_len] + noise

# Set up training and measurement data
X_train = u_obs[:,0:train_len]
X_meas = u_obs[:,train_len:train_len+forecast_len]

Now, using the [teaspoon](https://teaspoontda.github.io/teaspoon/modules/DAF/Forecasting.html) documentation and the ```random_feature_map_model``` function, generate the model coefficients for training the random feature map model using $D_r=300$ and the default random feature distribution parameters. For consistency, set ```seed=r_seed```.

### Answer Key (double click this cell):

---

<!-- W_LR, W_in, b_in = random_feature_map_model(X_train,Dr=300, seed=r_seed) -->

---

In [ ]:
# Train model
W_LR, W_in, b_in = # YOUR CODE HERE

Now that we have the trained model, use the ```get_forecast``` function from [teaspoon](https://teaspoontda.github.io/teaspoon/modules/DAF/Forecasting.html) to generate a forecast for the system starting at ```X_meas[:,0]```. Be sure to use the ```X_start``` array as the forecast starting point which is formatted to be of shape (3,1) as this function requires the input shape to be in that form for consistency with other forecast functions.

### Answer Key (double click this cell):

---

<!-- X_model= get_forecast(X_start, W_LR, mu=(W_in, b_in),forecast_len=forecast_len) -->

---

In [ ]:
# Generate forecast
X_start = X_meas[:,0].reshape(-1,1)

X_model= # YOUR CODE HERE

Now run the cell below to plot the forecasted system states.

In [ ]:
# Plot measurements and forecast
fig = plt.figure(figsize=(15, 5), dpi=200)
gs = GridSpec(1, 3)
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(X_model[0,:],'r', label="Forecast")
ax1.plot(X_meas[0,:], '.b', label="Measurement")
ax1.plot([],[])
ax1.set_title('x', fontsize='x-large')
ax1.tick_params(axis='both', which='major', labelsize='x-large')
ax1.set_ylim((-30,30))


ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(X_model[1,:],'r', label="Forecast")
ax2.plot(X_meas[1,:], '.b', label="Measurement")
ax2.plot([],[])
ax2.set_title('y', fontsize='x-large')
ax2.tick_params(axis='both', which='major', labelsize='x-large')
ax2.set_ylim((-30,30))

ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(X_model[2,:],'r', label="Forecast")
ax3.plot(X_meas[2,:], '.b', label="Measurement")
ax3.plot([],[])
ax3.legend(fontsize='large', loc='upper left')
ax3.set_title('z', fontsize='x-large')
ax3.tick_params(axis='both', which='major', labelsize='x-large')
ax3.set_ylim((0,60))

plt.tight_layout()
plt.show()

If you still have time, try to modify the random seed to see how the forecast varies with different random features. You can also try different reservoir dimensions ```Dr``` to see how that changes the result. I have also included functionality for doing forecasting with Long Short Term Memory (LSTM) networks on [teaspoon](https://teaspoontda.github.io/teaspoon/modules/DAF/Forecasting.html) so feel free to play around with that method too, but pay attention to the documentation differences for the LSTM method.

# STOP HERE 🛑

---

## Part III: Topological Data Assimilation (10 minutes)

For the final part, we can put the pieces together and optimize the forecast model to minimize topological differences with persistence optimization.

In [ ]:
# Import necessary libraries
import numpy as np
from teaspoon.MakeData.DynSysLib.autonomous_dissipative_flows import lorenz
from teaspoon.DAF.data_assimilation import TADA
from teaspoon.DAF.forecasting import forecast_time
from teaspoon.DAF.forecasting import random_feature_map_model
from teaspoon.DAF.forecasting import get_forecast
from teaspoon.DAF.forecasting import G_rfm
import tensorflow as tf
from IPython.display import clear_output
from matplotlib.patches import Rectangle

Since we already simulated the chaotic lorenz system with added noise and generated a random feature map model, we can start by initializing the necessary parameters for TADA. Namely, we need to set the optimization learning rate ```lr```, the learning rate decay rate ```d_rate```, the sliding ```window_size``` and the maximum number of TADA steps ```max_window_number```.

In [ ]:
# Set TADA parameters
lr = 1e-6
d_rate = 0.99
opt_params = [lr, d_rate]
window_size = 50
max_window_number = 100

Next, we set up the model weights as a ```tensorflow``` variable. This is not required for the LSTM model so see the example on [teaspoon](https://teaspoontda.github.io/teaspoon/modules/DAF/DataAssimilation.html) to see how this part changes. Each forecast function $G$ requires three inputs: $X_p$ a matrix of previous time points to use for predicting the next, $w$ a weight matrix or tensor for making the predictions and $\mu$ a list of internal parameters specific to the chosen forecast model. 

For the random feature map model, $\mu$ takes the form $(W_{in},b_{in})$. We also take $p=1$ for the random feature map model because it only uses the last point to predict the next. 

In [ ]:
# Optimization variable setup
W_opt = [tf.Variable(W_LR, trainable=True, dtype=tf.float64)]
mu = (W_in, b_in)
p=1

Now inside of the TADA loop I created, use the [TADA](https://teaspoontda.github.io/teaspoon/modules/DAF/DataAssimilation.html) function from teaspoon to compute new model parameters ```W_opt``` at the next assimilation window. All of the required TADA parameters have been set for you in previous cells, but for this example set ```j2_sample_size``` to 100. For this example, pass the full ```u_obs``` array. In practice you would update this as new measurements come in but since we have all of them it is easier just to use the full array. The TADA function only accesses measurements up to the current window number. 

Run the TADA optimization loop and watch how the forecast changes as more measurements are obtained. 

### Answer Key (double click this cell):

---

<!-- W_opt = TADA(u_obs, window_size, model_parameters=model_parameters, train_len=train_len, opt_params=opt_params, window_number=window_number, j2_sample_size=100) -->

---

In [ ]:
# TADA optimization loop
for window_number in range(1,max_window_number):
    # Update model parameters
    model_parameters=[W_opt, W_LR, mu, G_rfm, p]
    
    ####################################################################################################################################
    ####################################################################################################################################
    ####################################################################################################################################


    W_opt = # YOUR CODE HERE

    
    ####################################################################################################################################
    ####################################################################################################################################
    ####################################################################################################################################
    # Generate and plot forecasts at each optimization step
    # Set forecast parameters
    plot_fc_length = 200
    start = train_len
    end = train_len + plot_fc_length + 1

    # Forecast TADA and LR models and get measurements
    X_model_tada = get_forecast(X_meas[:,0].reshape(-1,1), W_opt, mu,forecast_len=end-train_len, G=G_rfm)
    X_model_lr =get_forecast(X_meas[:,0].reshape(-1,1), W_LR, mu,forecast_len=end-train_len, G=G_rfm)
    X_meas = u_obs[:,start:end]

    clear_output(wait=True)

    # Plot measurements and forecast
    fig = plt.figure(figsize=(15, 5), dpi=200)
    gs = GridSpec(1, 3)
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(X_model_tada[0,:],'g', label="TADA")
    ax1.plot(X_model_lr[0,:],'--r', label="LR")
    ax1.plot(X_meas[0,:], '.b', label="Measurement")
    # Add a rectangle to highlight a specific region
    rect = Rectangle((np.max([0.0, window_number-window_size]),-30), np.min([window_number, window_size]), 60, linewidth=2, edgecolor='none', facecolor='blue', alpha=0.2)
    ax1.add_patch(rect)
    ax1.plot([],[])
    ax1.set_title('x', fontsize='x-large')
    ax1.tick_params(axis='both', which='major', labelsize='x-large')
    ax1.set_ylim((-30,30))


    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(X_model_tada[1,:],'g', label="TADA")
    ax2.plot(X_model_lr[1,:],'--r', label="LR")
    ax2.plot(X_meas[1,:], '.b', label="Measurement")
    rect = Rectangle((np.max([0.0, window_number-window_size]),-30), np.min([window_number, window_size]), 60, linewidth=2, edgecolor='none', facecolor='blue', alpha=0.2)
    ax2.add_patch(rect)
    ax2.plot([],[])
    ax2.set_title('y', fontsize='x-large')
    ax2.tick_params(axis='both', which='major', labelsize='x-large')
    ax2.set_ylim((-30,30))

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.plot(X_model_tada[2,:],'g', label="TADA")
    ax3.plot(X_model_lr[2,:],'--r', label="LR")
    ax3.plot(X_meas[2,:], '.b', label="Measurement")
    rect = Rectangle((np.max([0.0, window_number-window_size]),0), np.min([window_number, window_size]), 60, linewidth=2, edgecolor='none', facecolor='blue', alpha=0.2)
    ax3.add_patch(rect)
    ax3.plot([],[])
    ax3.legend(fontsize='large', loc='upper left')
    ax3.set_title('z', fontsize='x-large')
    ax3.tick_params(axis='both', which='major', labelsize='x-large')
    ax3.set_ylim((0,60))

    plt.tight_layout()
    plt.show()
    ####################################################################################################################################
    ####################################################################################################################################
    ####################################################################################################################################

If you still have time, vary the learning rate and noise level to see how the TADA forecast is affected. You can also try the LSTM example in the [documentation](https://teaspoontda.github.io/teaspoon/modules/DAF/DataAssimilation.html). 